# Setup
Notebooks call reusable functions from `src/`.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.config import load_config, resolve_paths, set_global_seed, get_seed
config = load_config(ROOT / 'configs/project_config.yaml')
paths = resolve_paths(config)
set_global_seed(get_seed(config))
print('project root:', paths.root)


## Baselines

In [ ]:
from src.data_loader import load_parquet
from src.feature_engineering import get_model_feature_columns
from src.temporal_split import chronological_date_split, masks_from_split
from src.baselines import run_baselines
from src.reporting import save_json
df = load_parquet(paths.processed)
# Prefer labeled frame if present
try:
    from pathlib import Path
    p = paths.processed
    df = load_parquet(p)
except Exception:
    pass
df = df.sort_values('timestamp').reset_index(drop=True)
if 'label' not in df.columns:
    raise SystemExit('Run pipeline or notebook 03 first to create labels')
split = chronological_date_split(df, 0.6, 0.2, 0.2, 60)
masks = masks_from_split(len(df), split)
valid = df['label'].notna().to_numpy()
for k in masks: masks[k] &= valid
cols = get_model_feature_columns(df)
res = run_baselines(df, cols, masks, float(df['epsilon'].iloc[0]))
save_json(res, paths.metrics/'baseline_metrics.json')
print({k:v.get('macro_f1') for k,v in res.items() if isinstance(v, dict) and 'macro_f1' in v})
